# 从 Attention 到 TTT：线性化之路（5/5）—— 线性版本：DeltaNet 与 MesaNet

最后一册把 $\mu_W$ 简化成线性模型，看看 TTT 雏形具体落成哪两类常见结构，
以及它们状态大小的差别。

## 1. 线性化：$\mu_W(k) = Wk$

实践中通常把 TTT 的 $\mu_W(k)$ 简化为线性映射 $Wk$。此时损失是线性回归的平方误差：

$$\ell(W_{t-1}; k_t, v_t) = \tfrac{1}{2}\Vert v_t - W k_t\Vert^2$$

对应的流式更新规则（对 $W$ 求梯度）是：

$$W_t = W_{t-1} - \eta_t\,(W_{t-1}k_t - v_t)\,k_t^{\top}$$

配上读出 $o_t = W_t q_t$，就得到了 **DeltaNet**。

把 $\mu_W$ 限定为线性函数的好处：模型可以并行计算，并且只承担 token-mixer 的角色；
非线性部分交给 MLP 层和层堆叠。另外，对线性模型来说，第 4 册的平均场等式是**精确恒等**，
不再是近似。

In [ ]:
import numpy as np

rng = np.random.default_rng(4)
d, T = 3, 10
eta = 0.1

K = rng.standard_normal((T, d))
K /= np.linalg.norm(K, axis=1, keepdims=True)
V = rng.standard_normal((T, d))
q = rng.standard_normal(d)

# DeltaNet 式递推: 单个状态矩阵 W
W = np.zeros((d, d))
for t in range(T):
    pred = W @ K[t]
    W = W - eta * np.outer(pred - V[t], K[t])
o_delta = W @ q
print("DeltaNet 读出 o:", o_delta.round(3))
print("状态矩阵 W 规模: %d×%d = %d 个元素" % (d, d, d*d))

## 2. 解析解路线：不累加 W，累加 G 和 H

线性回归还可以直接求解析解，不走 SGD。定义两个累加器：

$$G_t = G_{t-1} + v_t k_t^{\top}, \qquad H_t = H_{t-1} + k_t k_t^{\top}, \qquad o_t = G_t H_t^{-1} q_t$$

- $G$：值与键的外积累加；
- $H$：键自相关累加；
- 读出时用 $G H^{-1} q$。

这就是传统的 **Vanilla Linear Attention（VaLA）**；给 $G$、$H$ 补上 Decay（衰减），
便得到 **MesaNet**。

In [ ]:
# VaLA / MesaNet 路线: 两个累加器 G, H
G = np.zeros((d, d))
H = np.zeros((d, d))
eye = np.eye(d)
decay = 0.95  # 给 H、G 补 decay 即 MesaNet; 取 1.0 即纯 VaLA

for t in range(T):
    G = decay * G + np.outer(V[t], K[t])
    H = decay * H + np.outer(K[t], K[t])

o_vala = G @ np.linalg.solve(H + 1e-6 * eye, q)
print("VaLA 读出 o:     ", o_vala.round(3))
print("累加器 G 规模:   %d×%d" % (d, d))
print("累加器 H 规模:   %d×%d" % (d, d))
print("状态元素总数:    %d (比单状态 W 多 1 倍)" % (2 * d * d))

## 3. DeltaNet vs MesaNet（VaLA）

| 项 | DeltaNet（delta rule） | MesaNet / VaLA（解析解累加器） |
|---|---|---|
| 状态 | 单个矩阵 $W$ | 两个矩阵 $G$、$H$ |
| 更新 | 一步梯度下降修正 $W$ | 直接外积累加 |
| 状态大小 | $d^2$ | $2d^2$（多约 50%） |
| 额外机制 | delta 改写 | 补 Decay 后为 MesaNet |

文章的判断：假设 MesaNet 与 DeltaNet 处于同一能力上限，VaLA 用更简单的累加规则，
代价是状态变量增加 50%——也就是说，**VaLA 只有把状态加大，才有机会追上 Delta 系列**。

## 4. 整条链回顾

沿着文章的"平均之路"，五类概念的关系可以这样串起来：

```
Softmax Attention (原式, 平方复杂度, 逐 token 缓存 KV)
        │  ① 分布推广: v → E[v|k], 消去 V Cache
        ▼
条件分布 μ_W(k) + 流式 SGD 更新 W
        │  ② 两次平均: μ(Σa k) ；Σa k ≈ λq
        ▼
TTT 形式的 RNN: 状态 W, 读出 o = μ_W(q)
        │  ③ μ 线性化
        ├── DeltaNet: 单状态 W, delta rule 递推
        └── VaLA / MesaNet: 双状态 G、H 累加, 加 Decay
```

每一步都是用一定的信息损失，换缓存体积或复杂度的下降；理解这条链，
就能看清各种线性注意力之间"谁是谁的特例"。

## 5. 小结（5/5）与延伸阅读

- $\mu_W(k)=Wk$ 线性化后，delta rule 递推给出 **DeltaNet**；
- 解析解累加器 $G$、$H$ 给出 **VaLA**，加 Decay 为 **MesaNet**，状态多 50%；
- 线性模型下平均场是精确恒等式；非线性交给 MLP 和层堆叠。

原文存档：`docs/attention-linearization-to-ttt.md`，文中引用：

1. *将 Softmax Attention 线性化为 Gated DeltaNet*: https://spaces.ac.cn/archives/11823
2. *TTT (Test-Time Training)*: https://papers.cool/arxiv/2407.04620
3. *MLA*: https://spaces.ac.cn/archives/11111
4. *DSV4*: https://papers.cool/arxiv/2606.19348
5. *DeltaNet*: https://papers.cool/arxiv/2406.06484
6. *MesaNet*: https://papers.cool/arxiv/2506.05233